In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Import data

In [2]:
DATA_PATH = Path("../data/train.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"File path: {DATA_PATH}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

Dataset loaded successfully.
File path: ../data/train.csv
Rows: 400
Columns: 2


,text,label
0,How does staking work and what rewards can I e...,general
1,"Hello team, The app won't let me sign in, it j...",account-access
2,"Hey, My withdrawal of 2 ETH shows completed bu...",transaction-dispute
3,How does staking work and what rewards can I e...,general
4,"Hi, How long do SOL withdrawals usually take t...",general


## Data Shape

In [3]:
print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset info:")
df.info()

Dataset shape:
(400, 2)

Column names:
['text', 'label']

Data types:
text     str
label    str
dtype: object

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    400 non-null    str  
 1   label   400 non-null    str  
dtypes: str(2)
memory usage: 6.4 KB


In [4]:
print("Random sample rows:")
display(df.sample(10, random_state=30))

Random sample rows:


,text,label
35,Can I convert SOL directly into another coin w...,general
316,"Hello team, How long do USDT withdrawals usual...",general
281,Do you support Bitcoin? I don't see it listed ...,general
74,Please help. Can you explain how to move SOL t...,general
296,"Please help. My withdrawal of $10,000 shows co...",transaction-dispute
337,"Hello team, Which countries is your service av...",general
235,Where can I learn more about how transaction f...,general
259,The fee on my last Cardano trade is much highe...,transaction-dispute
220,"Hi, What are the fees for buying and selling ETH?",general
155,I was logged out everywhere and the app won't ...,account-access


## Idnetify Null and Duplicates

In [5]:
print("Null values:")
print(df.isna().sum())

empty_text = (df["text"].astype(str).str.strip() == "").sum()
empty_label = (df["label"].astype(str).str.strip() == "").sum()

print("\nEmpty text rows:", empty_text)
print("Empty label rows:", empty_label)

if empty_text == 0 and empty_label == 0 and df.isna().sum().sum() == 0:
    print("\n No missing or empty values found.")
else:
    print("\n Missing/empty values found.")

Null values:
text     0
label    0
dtype: int64

Empty text rows: 0
Empty label rows: 0

 No missing or empty values found.


In [6]:
duplicate_rows = df.duplicated().sum()
duplicate_texts = df.duplicated(subset=["text"]).sum()

print("Duplicate full rows:", duplicate_rows)
print("Duplicate text rows:", duplicate_texts)

if duplicate_rows == 0 and duplicate_texts == 0:
    print("\nNo duplicate rows/texts found.")
else:
    print("\nDuplicates found.")

Duplicate full rows: 0
Duplicate text rows: 0

No duplicate rows/texts found.


## Class destribution and Sample data by class

In [11]:
class_counts = df["label"].value_counts()
class_percentages = (df["label"].value_counts(normalize=True) * 100).round(2)

print("Class counts:")
print(class_counts)


Class counts:
label
general                160
account-access         100
transaction-dispute     90
fraud-report            50
Name: count, dtype: int64


In [12]:
for label, group in df.groupby("label"):
    print(f"\n===== {label} =====")
    for text in group["text"].sample(3, random_state=30):
        print("-", text)




===== account-access =====
- My 2FA isn't working, I got a laptop and can't receive the verification code. Let me know.
- Hello team, Face ID stopped working on the phone and now I'm stuck at the login prompt. This is time sensitive.
- The app won't let me sign in, it just spins on the login screen. Please advise.

===== fraud-report =====
- I think my account was hacked, there are trades on it I never made. Thanks.
- Hello team, I see a Android app I don't recognize on my account and money was moved out.
- Please help. There's a suspicious SOL transfer to an unknown wallet that I did not make. Please advise.

===== general =====
- Please help. What documents do I need to raise my account limits? This is time sensitive.
- Is there an app for the work computer? How do I get started? Thanks.
- Please help. How do I add a beneficiary or change my account details? This is time sensitive.

===== transaction-dispute =====
- Quick question, My withdrawal of $15,000 shows completed but I neve

## Top words and top bigrams by class

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_terms(texts, ngram_range=(1, 1), top_n=15):
    """
    Returns top words or n-grams from a list/Series of texts.
    
    ngram_range=(1, 1) -> single words
    ngram_range=(2, 2) -> bigrams / two-word phrases
    min_df=2 -> Keep a word if it appears in at least 2 different messages.
    """
    vectorizer = CountVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=ngram_range,
        min_df=2
    )

    X = vectorizer.fit_transform(texts)
    counts = X.sum(axis=0).A1
    vocab = vectorizer.get_feature_names_out()

    top_indices = counts.argsort()[::-1][:top_n]

    return [(vocab[idx], int(counts[idx])) for idx in top_indices]

In [14]:
for label in sorted(df["label"].unique()):
    texts = df.loc[df["label"] == label, "text"].astype(str)
    top_words = get_top_terms(texts, ngram_range=(1, 1), top_n=20)

    print(f"\n===== Top words for {label} =====")
    for word, count in top_words:
        print(f"{word}: {count}")




===== Top words for account-access =====
login: 35
help: 33
account: 31
reset: 24
code: 23
password: 23
access: 22
need: 21
time: 21
let: 21
sensitive: 21
app: 20
working: 19
verification: 17
keeps: 16
know: 16
quick: 16
question: 16
thanks: 13
log: 13

===== Top words for fraud-report =====
account: 25
help: 18
team: 11
know: 10
don: 10
time: 9
sensitive: 9
hello: 8
quick: 7
question: 7
appreciate: 7
think: 7
email: 7
hi: 7
urgent: 7
fraud: 7
wallet: 7
support: 7
moved: 6
money: 6

===== Top words for general =====
help: 37
advise: 26
hi: 25
sensitive: 25
time: 25
quick: 21
question: 21
fees: 20
documents: 18
let: 18
know: 18
hello: 17
urgent: 17
team: 17
appreciate: 16
app: 16
account: 16
hey: 16
thanks: 16
external: 14

===== Top words for transaction-dispute =====
help: 30
refund: 26
transaction: 22
want: 22
shows: 21
order: 19
appreciate: 19
twice: 17
buy: 17
days: 15
received: 15
confirmed: 14
balance: 14
sensitive: 13
time: 13
purchase: 13
hey: 13
fee: 12
charged: 12
quick: 11


In [15]:
for label in sorted(df["label"].unique()):
    texts = df.loc[df["label"] == label, "text"].astype(str)
    top_bigrams = get_top_terms(texts, ngram_range=(2, 2), top_n=15)

    print(f"\n===== Top bigrams for {label} =====")
    for bigram, count in top_bigrams:
        print(f"{bigram}: {count}")




===== Top bigrams for account-access =====
time sensitive: 21
quick question: 16
let know: 16
factor authentication: 12
app won: 11
hello team: 11
receive verification: 10
isn working: 10
2fa isn: 10
working got: 10
verification code: 10
face id: 9
log account: 9
access login: 9
password wrong: 9

===== Top bigrams for fraud-report =====
time sensitive: 9
hello team: 8
think account: 7
appreciate help: 7
quick question: 7
don recognize: 6
money moved: 6
let know: 6
account money: 6
recognize account: 6
account authorized: 5
report fraud: 4
phishing email: 4
pretending support: 4
entered credentials: 4

===== Top bigrams for general =====
time sensitive: 25
quick question: 21
let know: 18
hello team: 17
appreciate help: 16
external wallet: 14
minimum buy: 14
price alerts: 12
usually process: 12
enable price: 12
withdrawals usually: 12
tax documents: 11
documents year: 11
download tax: 11
buying selling: 10

===== Top bigrams for transaction-dispute =====
appreciate help: 19
time sensit